# RQ4 — Modality Sensitivity

Fig. 6 (mean final IoU by model, scene type and manipulation regime) and Fig. 7 (IoU reduction over the image/text budget split of multi-modal test cases).

In [ ]:
import sys

sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from experiments.analysis.loader import load_all_results, success_only
from experiments.analysis.reporting import setup_matplotlib, tex

setup_matplotlib()

df_all = load_all_results(include_baseline_fail=True)
df = success_only(df_all)

MODELS = [m for m in sorted(df.model.unique()) if (df.model == m).sum() >= 30]

MODEL_LABEL = {
    "qwen": "Qwen3-VL",
    "kimi": "Kimi-VL",
    "intern": "InternVL-3.5",
    "gemma": "Gemma3-4B",
    "deepseek": "Deepseek-VL2",
    "nemotron": "Nemotron3-ON",
}

SCENE_LABEL = {
    "single/solo": "SC-SI",
    "single/multi": "SC-MI",
    "multi": "MC",
    "udacity": "Driving",
}
SCENES = ["single/solo", "single/multi", "multi", "udacity"]


def mlabel(m):
    return MODEL_LABEL.get(m, tex(m))


print("Models:", [mlabel(m) for m in MODELS])

## Fig. 6 — Mean final IoU by model, scene type and manipulation regime

Cells with fewer than 3 runs are left empty.

In [ ]:
GMODES = ["multi", "image", "text"]
GMODE_LABEL = {"multi": "Multimodal", "image": "Image-only", "text": "Text-only"}
nr, nc = len(SCENES), len(GMODES)

n = len(MODELS)
CELL = 2.6
CB_W = 0.18

fig, axes = plt.subplots(
    1,
    n + 1,
    figsize=(n * CELL + CELL * CB_W, CELL),
    gridspec_kw={"width_ratios": [1] * n + [CB_W], "wspace": 0.02},
)
data_axes, ax_cb = axes[:n], axes[n]

for k, (ax, model) in enumerate(zip(data_axes, MODELS)):
    matrix = np.full((nr, nc), np.nan)
    for i, scene in enumerate(SCENES):
        for j, gmode in enumerate(GMODES):
            sub = df[(df.model == model) & (df.obj_category == scene) & (df.genome_mode == gmode)]
            if len(sub) >= 3:
                matrix[i, j] = sub["final_iou"].mean()

    im = ax.imshow(matrix, cmap="RdYlGn", vmin=0, vmax=0.5, aspect="equal")
    ax.set_aspect("equal")
    if k == 0:
        ax.set_yticks(range(nr))
        ax.set_yticklabels([SCENE_LABEL[s] for s in SCENES], fontsize=16)
    else:
        ax.set_yticks([])
    ax.set_xticks(range(nc))
    ax.set_xticklabels([GMODE_LABEL[g] for g in GMODES], rotation=30, ha="right", fontsize=14)
    ax.set_title(mlabel(model), fontsize=18, pad=4)
    for i in range(nr):
        for j in range(nc):
            if not np.isnan(matrix[i, j]):
                ax.text(
                    j,
                    i,
                    f"{matrix[i,j]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=14,
                    color="k",
                    fontweight="bold",
                )

plt.colorbar(im, cax=ax_cb)
ax_cb.set_ylabel("Mean final IoU", fontsize=16)
ax_cb.tick_params(labelsize=14)

plt.savefig("figures/rq4_scene_modality_heatmap.pdf", dpi=150, bbox_inches="tight")
plt.show()

## Fig. 7 — IoU reduction as a function of the image and text budgets (multi-modal runs)

Gaussian-weighted (σ = 0.03) average of the per-run IoU reduction; opacity reflects point density. The hatched triangle is outside the budget B = 1.

In [ ]:
from matplotlib.patches import Polygon
sub_mm = df[(df["genome_mode"] == "multi") & df["model"].isin(MODELS)]
sub_mm = sub_mm.dropna(subset=["img_budget_used", "txt_budget_used"])

GS = 500
SIGMA = 0.03
xi = np.linspace(0, 1, GS)
yi = np.linspace(0, 1, GS)
gx, gy = np.meshgrid(xi, yi)

n = len(MODELS)
CELL = 3.8
CB_W = 0.18

fig, axes = plt.subplots(
    1,
    n + 1,
    figsize=(n * CELL + CELL * CB_W, CELL),
    gridspec_kw={"width_ratios": [1] * n + [CB_W]},
)
data_axes, ax_cb = axes[:n], axes[n]

for k, (ax, model) in enumerate(zip(data_axes, MODELS)):
    sub = sub_mm[sub_mm["model"] == model]
    px = sub["img_budget_used"].values
    py = sub["txt_budget_used"].values
    val = sub["iou_reduction"].values

    weighted_sum = np.zeros((GS, GS))
    weight_total = np.zeros((GS, GS))
    for xp, yp, v in zip(px, py, val):
        w = np.exp(-((gx - xp) ** 2 + (gy - yp) ** 2) / (2 * SIGMA**2))
        weighted_sum += w * v
        weight_total += w

    color = np.where(weight_total > 0, weighted_sum / weight_total, 0.0)
    pos_w = weight_total[weight_total > 0]
    if pos_w.size == 0:
        alpha = np.zeros_like(weight_total)
    else:
        alpha_scale = np.percentile(pos_w, 90)
        if alpha_scale <= 0:
            alpha = np.zeros_like(weight_total)
        else:
            alpha = np.clip(weight_total / alpha_scale, 0, 1)

    norm = plt.Normalize(vmin=0, vmax=1)
    rgba = plt.cm.RdYlGn_r(norm(color))
    rgba[..., 3] = alpha

    ax.imshow(rgba, origin="lower", extent=[0, 1, 0, 1], aspect="equal")

    invalid_region = Polygon(
        [(0, 1), (1, 1), (1, 0)],
        closed=True,
        facecolor="white",
        edgecolor="none",
        zorder=3,
        )
    ax.add_patch(invalid_region)

    # Add diagonal stripes over the masked region
    hatched_region = Polygon(
        [(0, 1), (1, 1), (1, 0)],
        closed=True,
        facecolor="none",
        edgecolor="gray",
        hatch="///",
        linewidth=0,
        zorder=4,
    )
    ax.add_patch(hatched_region)
    ax.plot([0, 0.5], [0, 0.5],color="black", alpha=0.5, linewidth=1, zorder=10)
    ax.plot([0, 1], [1, 0],color="black", linewidth=1)
    ax.set_aspect("equal")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_zorder(10)
    if k == 0:
        ax.set_ylabel("Text budget used", fontsize=20)
    ax.set_title(mlabel(model), fontsize=20, pad=4)
fig.supxlabel("Image budget used", fontsize=20)
sm = plt.cm.ScalarMappable(cmap="RdYlGn_r", norm=norm)
sm.set_array([])
plt.colorbar(sm, cax=ax_cb)
ax_cb.set_ylabel("\% IoU reduction")

plt.savefig("figures/rq4_budget_heatmap.pdf", dpi=150, bbox_inches="tight")
plt.show()